# 1. **라이브러리 임포트 및 환경 설정**

In [ ]:
!pip install pypdf
!pip install -U langchain-openai langchain-core
!pip install -U langchain-text-splitters

In [ ]:
import os
import re
import json
import getpass
import numpy as np
from google.colab import drive
from pypdf import PdfReader
from typing import List, Dict
from sklearn.metrics.pairwise import cosine_similarity

# LangChain 및 AI 모델 관련 임포트
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# ==============================================================================
# 1. 환경 설정 및 유틸리티 함수
# ==============================================================================

# 1) Google Drive 마운트
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2) OpenAI API 키 입력
if "OPENAI_API_KEY" not in os.environ:
    print("\n🔑 OpenAI API Key를 입력하세요 (입력 시 키가 화면에 노출되지 않습니다):")
    os.environ["OPENAI_API_KEY"] = getpass.getpass()

# 3) 데이터셋 루트 경로 설정
# 예: "/content/drive/MyDrive/your_project_folder/dataset"
DATASET_ROOT = "YOUR_GOOGLE_DRIVE_DATASET_PATH"

# 4) LLM 분석 엔진 초기화 (GPT-4o 모델 기반 고정)
try:
    llm = ChatOpenAI(model_name="gpt-4o", temperature=0, seed=42)
    print("✅ LLM (gpt-4o) 분석 엔진 초기화 성공")
except Exception as e:
    print(f"❌ LLM 초기화 실패: {e}")

# 5) 정형 데이터 추출을 위한 JSON 파서
def robust_json_parse(response_text):
    """LLM 응답 문맥에서 정형 JSON 객체만 정교하게 슬라이싱 및 추출"""
    try:
        text = response_text.replace("```json", "").replace("```", "").strip()
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            text = match.group(0)
        return json.loads(text)
    except:
        return None

# 6) PDF 텍스트 스트림 추출 유틸리티
def extract_text_from_pdf(pdf_path):
    """PyPDF를 활용한 특허 명세서 PDF 내부 텍스트 추출"""
    try:
        reader = PdfReader(pdf_path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() or ""
        return text
    except Exception as e:
        print(f"    ⚠️ PDF 읽기 오류: {e}")
        return ""

# 2. **구성요소 추출 및 구조화** - AI 전문 심사관 버전

In [2]:
# ==============================================================================
# 2. 구성요소 추출 및 구조화 (독립항 식별 + 앵커 분석 + 본문 매칭)
# ==============================================================================

def get_independent_claims(text: str, llm_client) -> List[Dict]:
    """정규식으로 항을 분리하고, LLM이 독립항 여부를 판별"""
    print("    [1단계] LLM 기반 독립항 식별 프로세스 시작...")

    # 1. 청구범위 섹션 추출
    claim_section_keywords = ["【청구범위】", "【특허청구범위】", "[청구범위]"]
    start_idx = -1
    for keyword in claim_section_keywords:
        idx = text.find(keyword)
        if idx != -1: start_idx = idx; break
    claims_full_text = text[start_idx:] if start_idx != -1 else text

    # 2. 정규식으로 항 단위 '단순 분할' (내용 수집용)
    header_pattern = re.compile(r"(?:【|\[)\s*청구항\s*(\d+)\s*(?:】|\])")
    matches = list(header_pattern.finditer(claims_full_text))

    raw_claims = []
    for i in range(len(matches)):
        c_num = matches[i].group(1)
        start = matches[i].end()
        end = matches[i+1].start() if i+1 < len(matches) else len(claims_full_text)
        content = claims_full_text[start:end].strip()
        if len(content) > 10:
            raw_claims.append({"num": c_num, "content": content})

    # 3. LLM에게 독립항만 골라달라고 요청
    independent_claims = []
    claims_summary = "\n".join([f"청구항 {c['num']}: {c['content'][:150]}..." for c in raw_claims])

    prompt = f"""
    아래는 특허 청구범위의 각 항 내용이다.
    이 중에서 다른 항을 인용하지 않는 '독립항'의 번호만 골라라.

    [판단 기준]
    - 독립항: 다른 항(예: 제1항, 청구항 1)을 인용하며 시작하지 않는 항.
    - 종속항: "제1항에 있어서", "제1항 또는 제2항에 따른" 처럼 다른 항의 번호를 언급하며 시작하는 항.

    [청구항 리스트]
    {claims_summary}

    반드시 JSON 배열 형식으로 답변해라: ["1", "5", "10"]
    """

    try:
        # 인자로 받은 llm_client 사용으로 결합도 통일
        response = llm_client.invoke([HumanMessage(content=prompt)])
        ind_nums = re.findall(r'"(\d+)"|\'(\d+)\'|(\d+)', response.content)
        ind_nums = [n for t in ind_nums for n in t if n]

        # 4. 식별된 번호에 해당하는 항만 리턴
        for claim in raw_claims:
            if claim['num'] in ind_nums:
                independent_claims.append({
                    "header": f"【청구항 {claim['num']}】",
                    "content": claim['content']
                })
                print(f"      ✅ LLM 식별 독립항: 【청구항 {claim['num']}】")

    except Exception as e:
        print(f"    ⚠️ LLM 식별 중 오류 발생: {e}. 첫 번째 항을 기본값으로 사용합니다.")
        if raw_claims:
            independent_claims.append({
                "header": f"【청구항 {raw_claims[0]['num']}】",
                "content": raw_claims[0]['content']
            })

    return independent_claims


def extract_integrated_elements(independent_claims: List[Dict], llm_client) -> List[Dict]:
    """AI 특화 심사관 관점으로 데이터 흐름 및 기술적 단계 중심의 구성요소 추출"""

    combined_claims_text = "\n\n".join([f"[{c['header']}]: {c['content']}" for c in independent_claims])

    system_prompt = """
    당신은 인공지능(AI) 및 소프트웨어 발명 전문 특허 심사관입니다.
    독립항에 기재된 기술적 사상을 다음의 [추출 기준]에 의거하여 핵심 구성요소로 구조화하십시오.

    [추출 기준]
    1. 프로세스 지향성: 하드웨어 명칭보다는 데이터의 처리 과정, 알고리즘의 실행 단계, 또는 시계열적 운영 프로세스를 하나의 구성요소 단위로 설정한다.
    2. 기술적 독립성: 입력 데이터의 정의, 전처리 로직, 모델의 학습 메커니즘, 추론 및 결과 생성, 최종 판단 알고리즘 등 기술적으로 구별되는 기능적 단계를 분리하여 추출한다.
    3. 문언의 충실성 및 명칭 정의: 구성요소의 명칭(component)은 청구범위에 기재된 용어를 우선적으로 사용한다.
       - 원문에서 '~부', '~기', '~장치' 등 장치 중심으로 기재된 경우 해당 명칭을 유지한다.
       - 원문에서 '~단계', '~방법' 등 공정 중심으로 기재된 경우 해당 명칭을 유지한다.
       - 목적과 수단이 드러나도록 원문의 용어를 조합하여 명사구 형태로 정의하되, 원문의 어미(부/단계 등)를 임의로 변경하지 않는다.
    4. 동작 중심의 앵커링: 각 구성요소가 성립하기 위한 핵심 서술어(anchor)는 해당 단계에서 수행되는 기술적 연산이나 물리적 동작을 나타내는 단어로 추출한다.
    5. 기능적 역할 정의: 각 구성요소가 전체 시스템 내에서 담당하는 기술적 기여도와 입출력 데이터의 관계를 역할(role)로 명시한다.
    6. 중복 배제 및 통합: 서로 다른 독립항에 기재되어 있더라도 기술적 실체와 수행 단계가 동일한 경우에는 하나의 구성요소로 통합하여 리스트를 구성한다.

    반드시 아래 JSON 배열 형식으로만 답변하세요:
    [
      {"component": "구성요소 명칭", "anchor": "핵심 로직/동작", "role": "기술적 목적 및 기여도"}
    ]
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"다음 독립항들로부터 위 기준에 따라 기술적 단계 중심의 구성요소를 추출하십시오:\n\n{combined_claims_text}")
    ]

    print(f"    [2단계] AI 전문 심사관 기준 구성요소 추출 중...")

    try:
        response = llm_client.invoke(messages)
        clean_text = re.sub(r'```json|```', '', response.content).strip()
        elements = json.loads(clean_text)
        return elements
    except Exception as e:
        print(f"    ⚠️ 구성요소 추출 중 오류 발생: {e}")
        return []


def get_evidence_spans(element_info, full_text: str, embedding_model, llm_client) -> str:
    # 1. 준비 과정
    comp_name = element_info.get('component', '')
    clean_name = re.sub(r'\(\d+\)', '', comp_name).strip()
    core_keyword = clean_name.replace(" ", "")

    # 2. 후보 문단 수집 및 규칙 기반 자체 스코어링
    paragraphs = re.split(r'([\[【]\d{4}[\]】])', full_text)
    scored_candidates = []

    for i in range(1, len(paragraphs), 2):
        p_id, p_text = paragraphs[i], paragraphs[i+1].strip()
        text_no_space = p_text.replace(" ", "")

        if core_keyword in text_no_space:
            score = 0
            # [규칙 1] 정의형 어미가 포함되어 있는가?
            if any(term in p_text for term in ["은 ", "는 ", "특징으로", "구성된다", "포함한다"]):
                score += 5
            # [규칙 2] 명칭 바로 뒤에 도면부호가 붙어 있는가?
            if re.search(re.escape(clean_name) + r'\s*\(\d+\)', p_text):
                score += 10
            # [규칙 3] 문단의 시작 부분에 명칭이 등장하는가?
            if p_text.startswith(clean_name) or p_text.startswith("상기 " + clean_name):
                score += 7

            scored_candidates.append({"score": score, "content": f"{p_id} {p_text}"})

    # 3. 점수 순으로 정렬하여 상위 10개 추출
    scored_candidates.sort(key=lambda x: x['score'], reverse=True)
    top_candidates = [c['content'] for c in scored_candidates[:10]]

    if not top_candidates:
        return "본문 내 명칭 일치 문단 없음"

    # 4. LLM 반추를 통한 최적 근거 1건 정밀 추출
    candidate_context = "\n\n".join(top_candidates)

    prompt = f"""
    당신은 특허 분석 전문가입니다.
    제시된 후보 문단들 중에서 구성요소 '{clean_name}'에 대한 '정의'나 '구체적인 기능 설명'이 가장 잘 나타난 문단 하나만 선택해주세요.

    [후보 문단들]
    {candidate_context}

    [선택 기준]
    1. '{clean_name}'이 문장의 주어로 사용되어 그 기능이나 단계를 설명하는 문단.
    2. 도면부호(예: S40, 110 등)가 함께 기재된 문단 우선.
    3. 단순히 명칭이 나열된 곳이 아닌, 기술적 실체가 상세히 설명된 곳.

    반드시 선택한 문단의 [번호]와 전체 내용을 그대로 출력하세요. 부가적인 설명은 하지 마세요.
    """

    try:
        response = llm_client.invoke([HumanMessage(content=prompt)])
        return response.content.strip()
    except Exception as e:
        # 💡 [교정] 변수명 에러 방지 (candidates -> top_candidates)
        return top_candidates[0]


def process_patent_structure(target_text: str, embedding_model, llm_client):
    """최종 파이프라인: 독립항 식별 -> 통합 구성요소 추출 -> 본문 근거 매칭"""
    print("\n🚀 특허 구조화 분석 프로세스 시작")

    # 1. 독립항 식별
    ind_claims = get_independent_claims(target_text, llm_client)
    if not ind_claims:
        return []

    # 2. 통합 구성요소 추출
    integrated_elements = extract_integrated_elements(ind_claims, llm_client)

    # 3. 본문 근거 매칭
    print(f"    [3단계] {len(integrated_elements)}개 구성요소별 본문 근거 매칭 중...")
    refined_elements = []
    for elem in integrated_elements:
        evidence = get_evidence_spans(elem, target_text, embedding_model, llm_client)
        elem['evidence_span'] = evidence
        refined_elements.append(elem)

    # 리포트 구조화
    report = {
        "independent_claims_count": len(ind_claims),
        "claim_ids": [c['header'] for c in ind_claims],
        "elements": refined_elements
    }

    return [report]

# 3. **[신규성 판단 로직]** 동의어 확장 신규성 판단 + 원문 근거 추출 + 문단 번호 매핑

In [3]:
def check_novelty_with_synonyms(target_elements, ref_db, embedding_model):
    """
    동의어 확장 검색 및 문단 번호 추출 기능이 강화된 신규성 리스크 분석
    """
    print("\n   [Phase 1] 지능형 신규성 리스크 분석 시작")

    analysis_results = {}

    for ref_name, data in ref_db.items():
        if not ref_name.startswith("ref_"): continue
        print(f"    🔎 선행문헌 정밀 분석 중: {ref_name} ...")

        ref_text = data.get('text', "")

        # 1. 문단 번호 패턴([0001])을 기준으로 텍스트 분리 및 매핑
        raw_chunks = re.split(r'(\[\d{4}\])', ref_text)
        ref_chunks = []
        current_id = "[0000]"

        for part in raw_chunks:
            part = part.strip()
            if not part: continue
            if re.match(r'\[\d{4}\]', part):
                current_id = part # 문단 번호 업데이트
            else:
                ref_chunks.append(f"{current_id} {part}")

        if not ref_chunks: continue

        # 청크 임베딩 생성
        chunk_embeddings = embedding_model.encode(ref_chunks)

        ref_score_reports = []
        total_scores = []

        for elem in target_elements:
            comp_name = elem['component']
            comp_role = elem['role']
            comp_anchor = elem['anchor']

            # 1. LLM을 이용한 기술적 동의어 확장
            synonym_prompt = f"특허 문헌에서 '{comp_name}'(역할: {comp_role})과 기술적으로 동일한 의미로 사용될 수 있는 대체 용어 3가지를 쉼표로 구분해서 말해줘."
            try:
                synonyms_res = llm.invoke([HumanMessage(content=synonym_prompt)]).content
                search_terms = [comp_name] + [s.strip() for s in synonyms_res.split(',')]
            except:
                search_terms = [comp_name]

            # 2. 하이브리드 검색 정밀도 향상
            collected_context = []
            query_text = f"명칭: {comp_name}, 동작: {comp_anchor}, 역할: {comp_role}"
            query_vec = embedding_model.encode([query_text])[0]

            similarities = np.dot(chunk_embeddings, query_vec) / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_vec))
            top_indices = similarities.argsort()[-4:][::-1]

            for idx in top_indices:
                collected_context.append(ref_chunks[idx])

            # 유의어 기반 텍스트 매칭
            for term in search_terms:
                if len(term.strip()) < 2: continue
                for chunk in ref_chunks:
                    if term in chunk and chunk not in collected_context:
                        collected_context.append(chunk)
                    if len(collected_context) >= 6: break
                if len(collected_context) >= 6: break

            context_text = "\n".join(collected_context)

            # 3. LLM 스코어링
            scoring_prompt = f"""
            당신은 전문 특허 심사관입니다. [타겟 구성요소]를 [선행문헌 문맥]과 대조하여 신규성을 심사하십시오.

            ### 📝 [대조 분석 대상]
            - 타겟 명칭: {comp_name}
            - 핵심동작(앵커): {comp_anchor}
            - 기술적 역할: {comp_role}

            ### 🎯 [정밀 매핑 가이드라인]
            1. **본질적 기능 탐색**: '발명을 실시하기 위한 구체적인 내용' 같은 단순 섹션 제목에 현혹되지 마십시오. 타겟의 **핵심동작({comp_anchor})과 역할({comp_role})이 실제로 구현되고 묘사된 가장 구체적인 단일 문단**을 특정해야 합니다.
            2. **개별성 유지**: 각 구성요소는 선행문헌 내에서 서로 다른 문단에 흩어져 있을 확률이 높습니다. 각 기능에 제일 알맞은 문단을 독립적으로 매핑하십시오.
            3. **유의어 유연성**: 명칭이 다르더라도 '{comp_name}'과 작용 효과가 동일한 유의어를 발견하면 근거로 채택하십시오.

            [선행문헌 문맥]
            {context_text}

            ### ⚖️ [심사 기준 및 출력 포맷]
            - Identical: 동작과 역할이 실질적으로 동일 (도면 부호 등이 포함된 구체적 문장 선호)
            - Modified: 취지는 유사하나 구체적 수단/방법에 변형이 있음
            - Distinct: 전체 문맥에서 해당 기능을 발견할 수 없음

            반드시 아래 JSON 형식으로만 답변하십시오:
            {{
              "component": "{comp_name}",
              "label": "Identical/Modified/Distinct",
              "score": <0에서 100 사이의 정수>,
              "reason": "해당 문단을 선정한 이유와 타겟과의 구체적인 기능 대조 결과를 상세히 기술",
              "original_quote": "발췌한 원문 1~2문장 (절대 중간에 끊지 말고 마침표로 끝나게 작성, 줄바꿈 제거)",
              "paragraph_id": "발췌한 원문이 속한 정확한 문단 번호 (예: [0035])"
            }}
            """

            try:
                res = llm.invoke([HumanMessage(content=scoring_prompt)]).content
                score_data = robust_json_parse(res)
                if score_data:
                    ref_score_reports.append(score_data)
                    total_scores.append(score_data['score'])
            except:
                continue

        if total_scores:
            avg_score = sum(total_scores) / len(total_scores)
            analysis_results[ref_name] = {
                "score_report": ref_score_reports,
                "overall_risk_score": avg_score
            }
            print(f"      👉 {ref_name} 리스크 지수: {avg_score:.1f}")

    if not analysis_results: return {"verdict": "분석 불가"}

    worst_ref = max(analysis_results, key=lambda k: analysis_results[k]['overall_risk_score'])
    worst_score = analysis_results[worst_ref]['overall_risk_score']
    verdict = "High Risk" if worst_score >= 85 else "Moderate Risk" if worst_score >= 50 else "Low Risk"

    return {
        "verdict": verdict,
        "primary_ref": worst_ref,
        "full_analysis": analysis_results
    }

# 4. **[진보성 판단 로직]** TSM 기반 분석

In [4]:
def check_inventive_step_v2(target_elements, novelty_report, ref_db, target_full_text):
    """
    TSM(Teaching-Suggestion-Motivation) 모델 기반 진보성 심층 분석
    """
    print("\n   [Phase 2] 진보성(TSM 모델) 논리적 결합 용이성 분석 시작")

    # 1. [3-1] 문맥 정보(Context) 추출 로직
    def extract_patent_sections(text):
        sections = {
            "기술분야": ["【기술분야】", "기술분야", "TECHNICAL FIELD"],
            "과제": ["【해결하려는 과제】", "발명이 해결하고자 하는 과제", "과제의 해결"],
            "수단": ["【과제의 해결 수단】", "과제의 해결수단", "Means for Solving"],
            "효과": ["【발명의 효과】", "발명의 효과", "ADVANTAGEOUS EFFECTS"]
        }
        extracted = {}
        for key, headers in sections.items():
            content = ""
            for h in headers:
                start = text.find(h)
                if start != -1:
                    # 다음 '【' 가 나올 때까지 추출
                    end = text.find("【", start + len(h))
                    content = text[start:end if end != -1 else len(text)].strip()
                    break
            extracted[key] = content if content else "섹션 추출 실패 (전체 텍스트 기반 분석 필요)"
        return extracted

    target_sections = extract_patent_sections(target_full_text)

    # 2. [3-2] 주인용(Primary) vs 부인용(Secondary) 설정
    primary_ref_name = novelty_report.get('primary_ref')
    primary_ref_data = ref_db.get(primary_ref_name, {}).get('text', "")

    # 부인용 문헌 (주인용을 제외한 나머지 중 차이점을 보완할 가능성이 있는 것)
    secondary_refs_context = ""
    for name, data in ref_db.items():
        if name != primary_ref_name:
            secondary_refs_context += f"\n[{name} 내용 요약]: {data.get('text', '')[:5000]}"

    # 3. [3-3, 3-4] 결합 용이성 및 효과의 현저성 통합 프롬프트
    tsm_prompt = f"""
    당신은 대한민국 특허청 수석 심사관입니다. 아래 [TSM 가이드라인]에 따라 진보성 부정 여부를 판단하세요.

    [1. 분석 대상 (Target)]
    - 구성요소: {json.dumps(target_elements, ensure_ascii=False)}
    - 해결과제: {target_sections['과제']}
    - 발명의 효과: {target_sections['효과']}

    [2. 선행문헌 (References)]
    - 주인용문헌 ({primary_ref_name}): {primary_ref_data[:10000]}
    - 부인용문헌 (기타): {secondary_refs_context}

    # ⚖️ [TSM 심사 가이드라인]
    3-3. 결합의 용이성 체크리스트 (Logic Gate):
    A. 통상의 창작 능력 (Design Change):
       단순 수치한정, 균등물 치환, 단순 설계 변경(배치 변경)에 해당하는가?
       (하나라도 YES면 진보성 부정 가능성 높음)
    B. 동기 유발 (Motivation):
       기술분야/과제가 공통되는가? 선행문헌에 결합의 시사/암시가 있는가?
       (YES가 많을수록 결합 용이 → 진보성 부정)

    3-4. 효과의 현저성 검토:
       차이점으로 인한 효과가 '이질적'이거나 '양적으로 현저'하여 예측 범위를 벗어나는가?
       (이 경우에만 예외적으로 진보성 인정 가능)

    반드시 아래 JSON 형식으로 답변하세요:
    {{
      "is_obvious": true/false,
      "primary_vs_secondary": "주인용문헌의 부족한 점을 어떤 부인용문헌이 채우는지 기술",
      "motivation_check": "결합 동기가 충분한지(분야/과제 공통성 등) 분석",
      "design_change_check": "단순 설계 변경이나 수치 한정에 불과한 요소가 있는지 적출",
      "effect_analysis": "효과의 현저성 인정 여부와 그 이유",
      "final_verdict": "진보성 부정(거절) 또는 인정"
    }}
    """

    try:
        response = llm.invoke([HumanMessage(content=tsm_prompt)]).content
        result = robust_json_parse(response)

        # 결과 출력
        print(f"\n   🔬 진보성 최종 판정: {result['final_verdict']}")
        print(f"    ├─ 🔗 결합 논리: {result['primary_vs_secondary'][:150]}...")
        print(f"    ├─ 💡 동기/설계: {result['motivation_check']} / {result['design_change_check']}")
        print(f"    └─ ✨ 효과 분석: {result['effect_analysis']}")

        return result
    except Exception as e:
        print(f"      ⚠️ 진보성 분석 중 오류: {e}")
        return {"final_verdict": "분석 실패"}

# 5. **메인 함수**

In [6]:
import os
import glob
import re
from sentence_transformers import SentenceTransformer

def main():
    """
    특허 리스크(신규성 및 진보성) 자동 평가 파이프라인 메인 실행 함수
    """
    # ==============================================================================
    # STEP 1. 분석 엔진 초기화 (Embedding & LLM)
    # ==============================================================================
    print("🔄 분석 엔진 가동 중 (Embedding & LLM)...")
    try:
        embedding_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
        llm_client = llm
    except Exception as e:
        print(f"❌ 초기화 실패: {e}"); return

    if not os.path.exists(DATASET_ROOT):
        print(f"❌ 데이터셋 경로를 찾을 수 없습니다: {DATASET_ROOT}"); return

    patent_folders = sorted([f.path for f in os.scandir(DATASET_ROOT) if f.is_dir()])

    # ==============================================================================
    # STEP 2. 특허 폴더 순회 및 타겟 출원서 분석
    # ==============================================================================
    for folder_path in patent_folders:
        folder_name = os.path.basename(folder_path)
        print("\n" + "="*100)
        print(f"📂 [종합 특허성 분석 리포트] 출원번호: {folder_name}")
        print("="*100)

        target_path = os.path.join(folder_path, "target.pdf")
        if not os.path.exists(target_path):
            print(f"⚠️ 'target.pdf' 파일 없음: {folder_name}"); continue

        target_text = extract_text_from_pdf(target_path)
        if not target_text.strip(): continue

        # 타겟 특허 구성요소 추출 및 구조화 (Component-Anchor-Role)
        analysis_result = process_patent_structure(target_text, embedding_model, llm_client)

        if not analysis_result or not analysis_result[0]['elements']:
            print("❌ 구성요소 추출 실패"); continue

        report = analysis_result[0]
        target_elements = report['elements']

        print(f"\n✅ 분석 완료: {report['independent_claims_count']}개 독립항 통합 분석 ({', '.join(report['claim_ids'])})")
        print("-" * 100)

        # 구조화된 타겟 구성요소 출력
        for idx, elem in enumerate(target_elements, 1):
            print(f"  {idx}. 【{elem['component']}】")
            print(f"     └─ ⚓ Anchor: {elem['anchor']}")
            print(f"     └─ 💡 Role: {elem['role']}")

            if elem.get('evidence_span') and elem['evidence_span'] != "근거 확인 불가":
                evidence_preview = elem['evidence_span'].split('\n\n[...]')[0]
                print(f"     └─ 🔍 evidence: {evidence_preview[:250].strip()}...")
            else:
                print(f"     └─ 🔍 evidence: 본문 내 매칭 내용을 찾지 못함")
            print()
        print("-" * 100)

        # ==============================================================================
        # STEP 3. 선행문헌(Reference) 데이터베이스 구축
        # ==============================================================================
        ref_db = {}
        ref_files = glob.glob(os.path.join(folder_path, "ref_*.pdf"))
        for rf in ref_files:
            ref_db[os.path.basename(rf)] = {"text": extract_text_from_pdf(rf)}
        print(f"2️⃣  선행문헌({len(ref_db)}개) 로딩 완료.")

        # ==============================================================================
        # STEP 4. 신규성(Novelty) 리스크 정밀 대조 분석
        # ==============================================================================
        print(f"3️⃣  신규성(Novelty) 리스크 분석 및 구성요소 대조 시작...")

        # 유의어 확장 및 핵심 앵커를 활용한 하이브리드 검색 기반 대조
        novelty_report = check_novelty_with_synonyms(target_elements, ref_db, embedding_model)

        primary_ref = novelty_report.get('primary_ref', 'N/A')
        print(f"\n   🔎 [신규성 상세 대조: {primary_ref}]")

        # 신규성 판정 결과 시각화
        if primary_ref in novelty_report.get('full_analysis', {}):
            details = novelty_report['full_analysis'][primary_ref]
            for item in details.get('score_report', []):
                label = item.get('label', 'Distinct')
                icon = "🔴" if label == "Identical" else "🟡" if label == "Modified" else "🟢"
                print(f"    {icon} [{item['component']}] -> {label} ({item.get('score', 0)}점)")
                print(f"       ├─ 위치: {item.get('paragraph_id', '미식별')}")
                print(f"       ├─ 근거: {item.get('reason', '')}")
                print(f"       └─ 원문: \"{item.get('original_quote', '')}\"\n")

        # ==============================================================================
        # STEP 5. TSM 모델 기반 진보성(Inventive Step) 추론
        # ==============================================================================
        print(f"4️⃣  진보성(Inventive Step) 결합 용이성 심층 분석...")
        inventive_report = check_inventive_step_v2(target_elements, novelty_report, ref_db, target_text)

        # ==============================================================================
        # STEP 6. 최종 종합 심사 의견서(Report) 출력
        # ==============================================================================
        print("\n" + "📊 [최종 종합 심사 의견]" + " " + "="*65)
        print(f"🚩 신규성(Novelty): {novelty_report.get('verdict', 'N/A')}")
        print(f"🚩 진보성(Inventive Step): {inventive_report.get('final_verdict', 'N/A')}")
        print("-" * 100)
        print(f"📝 [진보성 부정 논리 (TSM)]")
        print(f"   └─ 결합 동기: {inventive_report.get('motivation_check', '데이터 없음')}")
        print(f"   └─ 효과 분석: {inventive_report.get('effect_analysis', '데이터 없음')}")
        print("-" * 100)
        print(f"💡 분석 완료: {folder_name}")

if __name__ == "__main__":
    main()

🔄 분석 엔진 가동 중 (Embedding & LLM)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



📂 [종합 특허성 분석 리포트] 출원번호: 1020200121666

🚀 특허 구조화 분석 프로세스 시작
    [1단계] LLM 기반 독립항 식별 프로세스 시작...
      ✅ LLM 식별 독립항: 【청구항 1】
      ✅ LLM 식별 독립항: 【청구항 9】
    [2단계] AI 전문 심사관 기준 구성요소 추출 중...
    [3단계] 3개 구성요소별 본문 근거 매칭 중...

✅ 분석 완료: 2개 독립항 통합 분석 (【청구항 1】, 【청구항 9】)
----------------------------------------------------------------------------------------------------
  1. 【모델 학습 단계】
     └─ ⚓ Anchor: 학습시키는
     └─ 💡 Role: 압력 예측 모델을 학습시켜 사고 감지를 위한 기초 데이터 생성
     └─ 🔍 evidence: 【0081】 모델 학습 단계(S40)는 사고가 발생하지 않은 정상조건에서 감시구간에 포함된 복수의 측정지점의 압력과 유량, 펌프 가동상태, 전동밸브의 개도값이 학습데이터이고, 감시구간에 포함된 어느 하나의 측정지점의 압력이 라벨데이터인 학습데이터셋을 생성하는 학습데이터셋 생성 단계, 및 학습데이터셋으로 압력 예측 모델을 학습시키는 훈련단계를 포함할 수 있다....

  2. 【압력 예측 단계】
     └─ ⚓ Anchor: 예측하는
     └─ 💡 Role: 실시간 데이터를 기반으로 각 압력 측정지점의 압력을 예측하여 사고 감지에 활용
     └─ 🔍 evidence: 【0083】 압력 예측 단계(S50)는 사고감지 서버(100)의 사고감지부(140)에서 수행될 수 있 다. 압력 예측 단계(S50)는 상기 압력 예측 단계(S50)는 실시간으로 계측기(20)로부터 수 신되는 유량과 압력, 펌프의 가동상태, 전동밸브의 개도값을 입력데이터로 생성하는 입력 데이터 생성 단계, 및 입력데이터를 학습된 압력 예측 모델에 입력하여 측정